<a href="https://colab.research.google.com/github/va4756/deeplearn_practice/blob/main/pytorch_deeplearn_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 7.제 7 RNN 과 LSTM에 관한 내용

## 7.2.1 RNNCell

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# !pip install --user torchtext

In [3]:
import torch
# import torchtext
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

import time
import string
import re

In [4]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

In [5]:
"""
torchtext.legacy는 현재 더 이상 사용되지 않으므로, 최신 PyTorch에서는
**datasets + transformers + torch.utils.data.DataLoader**를 사용하는 것이 권장됩니다.
기존 코드와 동일한 기능(소문자 변환, 특수문자 제거, 길이 200으로 패딩, train/valid 분리,
batch 생성)을 수행하는 최신 버전 코드는 다음과 같습니다.
"""

'\ntorchtext.legacy는 현재 더 이상 사용되지 않으므로, 최신 PyTorch에서는\n**datasets + transformers + torch.utils.data.DataLoader**를 사용하는 것이 권장됩니다.\n기존 코드와 동일한 기능(소문자 변환, 특수문자 제거, 길이 200으로 패딩, train/valid 분리,\nbatch 생성)을 수행하는 최신 버전 코드는 다음과 같습니다.\n'

In [6]:
from datasets import load_dataset

# IMDB 데이터셋 로드
dataset = load_dataset(
    path="stanfordnlp/imdb"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [7]:
# train / test
train_data = dataset["train"]
test_data = dataset["test"]

In [8]:
# tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

MAX_LENGTH = 200

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [9]:
# 데이터 전처리
def preprocess(example):
    text = example["text"].lower()

    text = text.replace("<br />", " ")
    text = text.replace("<br", " ")

    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    text = re.sub(r"\s+", " ", text).strip()

    encoded = tokenizer(text,
                        max_length=MAX_LENGTH,
                        truncation=True,
                        padding="max_length")
    encoded["labels"] = example["label"]

    return encoded

In [10]:
train_data = train_data.map(preprocess)
test_data = test_data.map(preprocess)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [11]:
# train / valid 분리
split = train_data.train_test_split(test_size=0.2, seed=1)

train_data = split["train"]
valid_data = split["test"]

In [12]:
# PyTorch Tensor 형식 지정
columns = [
    "input_ids",
    "attention_mask",
    "labels"
]

train_data.set_format(type="torch", columns=columns)
valid_data.set_format(type="torch", columns=columns)
test_data.set_format(type="torch", columns=columns)

In [13]:
# DataLoader 생성
BATCH_SIZE = 64

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

In [25]:
# device 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

embedding_dim = 100
hidden_size = 300

In [26]:
# for batch in train_loader:
#     input_ids = batch["input_ids"].to(device)
#     attention_mask = batch["attention_mask"].to(device)
#     labels = batch["labels"].to(device)

In [27]:
from collections import Counter

counter = Counter()

for text in dataset["train"]["text"]:
    counter.update(text.split())

vocab = {"<pad>":0, "<unk>":1}

for word in counter:
    vocab[word] = len(vocab)

In [28]:
class RNNCell_Encoder(nn.Module):
    def __init__(self, input_dim, hidden_size):
        super(RNNCell_Encoder, self).__init__()
        self.rnn = nn.RNNCell(input_dim, hidden_size)

    def forward(self, inputs):
        bz = inputs.shape[1]
        ht = torch.zeros((bz, hidden_size)).to(device)

        for word in inputs:
            ht = self.rnn(word, ht)
        return ht

class Net(nn.Module):
    def __init__(self, ):
        super(Net, self).__init__()
        self.em = nn.Embedding(len(vocab), embedding_dim)
        self.rnn = RNNCell_Encoder(embedding_dim, hidden_size)
        self.fc1 = nn.Linear(hidden_size, 256)
        self.fc2 = nn.Linear(256, 3)

    def forward(self, x):
        x = self.em(x)
        x = self.rnn(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [30]:
model = Net()
model.to(device)

loss_fn = nn.CrossEntropyLoss()
optimr = torch.optim.Adam(model.parameters(), lr=0.0001)

In [32]:
def training(epoch, model, trainloader, validloader):
    correct = 0
    total = 0
    running_loss = 0

    model.train()
    for b in trainloader:
        x, y = b.text, b.label
        x, y = x.to(device), y.to(device)
        y_pred = model(x)
        loss = loss_fn(y_pred, y)
        optimr.zero_grad()
        loss.backward()
        optimr.step()
        with torch.no_grad():
            y_pred = torch.argmax(y_pred, dim=1)
            correct += (y_pred == y).sum().item()
            total += y.size(0)
            running_loss += loss.item()
    epoch_loss = running_loss / len(trainloader.dataset)
    epoch_acc = correct / total

    valid_correct = 0
    valid_total = 0
    valid_running_loss = 0

    model.eval()
    with torch.no_grad():
        for b in validloader:
            x, y = b.text, b.label
            x, y =